In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [ ]:
!git clone https://github.com/Break-Through-Tech/Market-Research-1B-discussion-intelligence-toolkit.git

fatal: destination path 'Market-Research-1B-discussion-intelligence-toolkit' already exists and is not an empty directory.


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving reddit-coarse-discourse-corpus.jsonl to reddit-coarse-discourse-corpus.jsonl


In [ ]:
from pathlib import Path
import sys

repo_root = Path("/content/Market-Research-1B-discussion-intelligence-toolkit")
corpus_path = Path("/content/reddit-coarse-discourse-corpus.jsonl")

print("Repository exists:", repo_root.is_dir())
print("Corpus exists:", corpus_path.is_file())

sys.path.insert(0, str(repo_root / "artifacts"))

from discourse_data import load_discourse_data

splits = load_discourse_data(corpus_path, seed=42)
print("Split sizes:", splits.summary())

Repository exists: True
Corpus exists: True
Split sizes: train=79517 dev=9373 test=10343


In [ ]:
import random
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert/distilbert-base-uncased"
)

rng = random.Random(42)
sample = rng.sample(splits.train, min(5000, len(splits.train)))

child_texts = [ex.text for ex in sample]
parent_texts = [ex.metadata.get("parent_text", "") for ex in sample]

child_tokens = tokenizer(
    child_texts,
    add_special_tokens=False,
    truncation=False,
)["input_ids"]

parent_tokens = tokenizer(
    parent_texts,
    add_special_tokens=False,
    truncation=False,
)["input_ids"]

child_lengths = np.array([len(ids) for ids in child_tokens])
parent_lengths = np.array([len(ids) for ids in parent_tokens])

pair_special_tokens = tokenizer.num_special_tokens_to_add(pair=True)
half_budget = (256 - pair_special_tokens) // 2

child_truncated = child_lengths > half_budget
parent_has_spare = parent_lengths < half_budget
could_recover_child_tokens = child_truncated & parent_has_spare

print("Sample size:", len(sample))
print("Current approximate budget per comment:", half_budget, "tokens")
print(
    "Children longer than their budget:",
    int(child_truncated.sum()),
    f"({child_truncated.mean():.1%})",
)
print(
    "Children that could use spare parent budget:",
    int(could_recover_child_tokens.sum()),
    f"({could_recover_child_tokens.mean():.1%})",
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (551 > 512). Running this sequence through the model will result in indexing errors


Sample size: 5000
Current approximate budget per comment: 126 tokens
Children longer than their budget: 571 (11.4%)
Children that could use spare parent budget: 399 (8.0%)


In [ ]:
parent_truncated = parent_lengths > half_budget
child_has_spare = child_lengths < half_budget

print(
    "Parents that could use spare child budget:",
    int((parent_truncated & child_has_spare).sum()),
    f"({(parent_truncated & child_has_spare).mean():.1%})",
)

print(
    "Children that could use spare budget when parent is empty:",
    int((could_recover_child_tokens & (parent_lengths == 0)).sum()),
)

print(
    "Children that could use spare budget with a nonempty parent:",
    int((could_recover_child_tokens & (parent_lengths > 0)).sum()),
)

Parents that could use spare child budget: 788 (15.8%)
Children that could use spare budget when parent is empty: 166
Children that could use spare budget with a nonempty parent: 233


In [ ]:
from pathlib import Path

transformer_path = (
    repo_root / "artifacts" / "discourse_transformer.py"
)

source = transformer_path.read_text(encoding="utf-8")

start_marker = (
    "            budget = "
    '(length - self.tokenizer.num_special_tokens_to_add(pair=True)) // 2'
)
end_marker = (
    "        else:\n"
    "            encoded = self.tokenizer(list(texts)"
)

if start_marker not in source or end_marker not in source:
    raise ValueError(
        "The encoder code differs from the expected version. "
        "No changes were made."
    )

start = source.index(start_marker)
end = source.index(end_marker, start)

replacement = """            # Share the token budget between parent and child.
            encoded = self.tokenizer(
                list(parents),
                list(texts),
                padding=True,
                truncation="longest_first",
                max_length=length,
                return_tensors="pt",
            )
"""

updated = source[:start] + replacement + source[end:]

backup_path = transformer_path.with_suffix(".py.original")
if not backup_path.exists():
    backup_path.write_text(source, encoding="utf-8")

transformer_path.write_text(updated, encoding="utf-8")

print("Updated the temporary Colab encoder.")
print("Original saved at:", backup_path)

Updated the temporary Colab encoder.
Original saved at: /content/Market-Research-1B-discussion-intelligence-toolkit/artifacts/discourse_transformer.py.original


In [ ]:
from discourse_transformer import DiscourseTransformer

encoder = object.__new__(DiscourseTransformer)
encoder.tokenizer = tokenizer
encoder.settings = {"max_length": 256, "parent_context": True}
encoder.device = "cpu"

def inspect_pair(parent, child):
    encoded = encoder._encode([child], [parent])
    ids = encoded["input_ids"][0].tolist()

    separators = [
        i for i, token_id in enumerate(ids)
        if token_id == tokenizer.sep_token_id
    ]

    assert len(separators) == 2
    parent_count = separators[0] - 1
    child_count = separators[1] - separators[0] - 1

    return parent_count, child_count, len(ids)

short_text = "hello " * 10
long_text = "word " * 200

cases = {
    "short parent, long child": (short_text, long_text),
    "long parent, short child": (long_text, short_text),
    "both long": (long_text, long_text),
    "empty parent, long child": ("", long_text),
}

for name, (parent, child) in cases.items():
    parent_count, child_count, total = inspect_pair(parent, child)
    print(
        f"{name}: parent={parent_count}, "
        f"child={child_count}, total={total}"
    )
    assert total <= 256

assert inspect_pair(short_text, long_text)[:2] == (10, 200)
assert inspect_pair(long_text, short_text)[:2] == (200, 10)
assert inspect_pair("", long_text)[:2] == (0, 200)

print("\nEncoder checks passed.")

short parent, long child: parent=10, child=200, total=213
long parent, short child: parent=200, child=10, total=213
both long: parent=126, child=127, total=256
empty parent, long child: parent=0, child=200, total=203

Encoder checks passed.


In [ ]:
import subprocess
import sys
import torch

assert torch.cuda.is_available(), "GPU is no longer available."

output_dir = Path("/content/shared_budget_experiment_01")

if output_dir.exists() and any(output_dir.iterdir()):
    raise FileExistsError(
        f"{output_dir} already contains files. "
        "Choose a new output directory rather than overwrite it."
    )

command = [
    sys.executable,
    str(repo_root / "scripts" / "train_discourse_transformer.py"),
    "--data", str(corpus_path),
    "--output", str(output_dir),
    "--parent-context",
    "--epochs", "3",
    "--batch-size", "16",
    "--learning-rate", "2e-5",
    "--max-length", "256",
    "--seed", "42",
    "--split-seed", "42",
]

print("Starting training on:", torch.cuda.get_device_name(0), flush=True)
subprocess.run(command, check=True)

Starting training on: Tesla T4


CompletedProcess(args=['/usr/bin/python3', '/content/Market-Research-1B-discussion-intelligence-toolkit/scripts/train_discourse_transformer.py', '--data', '/content/reddit-coarse-discourse-corpus.jsonl', '--output', '/content/shared_budget_experiment_01', '--parent-context', '--epochs', '3', '--batch-size', '16', '--learning-rate', '2e-5', '--max-length', '256', '--seed', '42', '--split-seed', '42'], returncode=0)

In [ ]:
import json

history_path = output_dir / "history.json"
compute_path = output_dir / "compute.json"

print("History exists:", history_path.is_file())
print("Compute report exists:", compute_path.is_file())
print("Best checkpoint exists:", (output_dir / "best").is_dir())

if history_path.is_file():
    history = json.loads(history_path.read_text(encoding="utf-8"))

    for epoch in history:
        dev = epoch["dev"]
        print(
            f"Epoch {epoch['epoch']}: "
            f"accuracy={dev['accuracy']:.4f}, "
            f"macro F1={dev['macro_f1']:.4f}, "
            f"weighted F1={dev['weighted_f1']:.4f}"
        )

if compute_path.is_file():
    compute = json.loads(compute_path.read_text(encoding="utf-8"))
    print(f"\nTraining time: {compute['elapsed_seconds'] / 60:.1f} minutes")
    print("Device:", compute["device"])

History exists: True
Compute report exists: True
Best checkpoint exists: True
Epoch 1: accuracy=0.7497, macro F1=0.5372, weighted F1=0.7374
Epoch 2: accuracy=0.7574, macro F1=0.5797, weighted F1=0.7491
Epoch 3: accuracy=0.7317, macro F1=0.5814, weighted F1=0.7346

Training time: 35.5 minutes
Device: Tesla T4


In [ ]:
import shutil
import zipfile
from google.colab import files

archive = shutil.make_archive(
    "/content/shared_budget_experiment_01",
    "zip",
    root_dir=output_dir,
)

with zipfile.ZipFile(archive, "a", compression=zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(
        transformer_path,
        arcname="code/discourse_transformer_shared_budget.py",
    )
    bundle.write(
        backup_path,
        arcname="code/discourse_transformer_original.py",
    )

print("Archive created:", archive)
files.download(archive)

Archive created: /content/shared_budget_experiment_01.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch
from torch import nn
from transformers import AutoModel

class GatedDiscourseModel(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.child_head = nn.Linear(hidden_size, num_labels)
        self.context_head = nn.Linear(hidden_size, num_labels)

        self.gate = nn.Linear(2 * hidden_size + 1, 1)

    def forward(self, child_inputs, context_inputs, has_parent):
        child_hidden = self.encoder(**child_inputs).last_hidden_state[:, 0]
        context_hidden = self.encoder(**context_inputs).last_hidden_state[:, 0]

        child_logits = self.child_head(child_hidden)
        context_logits = self.context_head(context_hidden)

        gate_input = torch.cat(
            [child_hidden, context_hidden, has_parent.reshape(-1, 1)],
            dim=1,
        )
        gate_value = torch.sigmoid(self.gate(gate_input))

        logits = (
            gate_value * context_logits
            + (1 - gate_value) * child_logits
        )

        return logits, gate_value

In [ ]:
import time
import torch
from torch import nn

torch.manual_seed(42)
device = torch.device("cuda")

gated_model = GatedDiscourseModel(
    model_name="distilbert/distilbert-base-uncased",
    num_labels=len(splits.labels),
).to(device)

batch = splits.train[:16]

children = [ex.text for ex in batch]
parents = [ex.metadata.get("parent_text", "") for ex in batch]

child_inputs = tokenizer(
    children,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt",
).to(device)

context_inputs = tokenizer(
    parents,
    children,
    padding=True,
    truncation="longest_first",
    max_length=256,
    return_tensors="pt",
).to(device)

has_parent = torch.tensor(
    [bool(parent.strip()) for parent in parents],
    dtype=torch.float32,
    device=device,
)

label_to_id = {label: i for i, label in enumerate(splits.labels)}
targets = torch.tensor(
    [label_to_id[ex.label] for ex in batch],
    dtype=torch.long,
    device=device,
)

gated_model.train()
gated_model.zero_grad()

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
start = time.perf_counter()

logits, gate_values = gated_model(
    child_inputs,
    context_inputs,
    has_parent,
)

loss = nn.CrossEntropyLoss()(logits, targets)
loss.backward()

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

print("Logit shape:", tuple(logits.shape))
print("Gate shape:", tuple(gate_values.shape))
print("Loss:", round(loss.item(), 4))
print("Forward/backward time:", round(elapsed, 2), "seconds")
print(
    "Peak allocated GPU memory:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GiB",
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Logit shape: (16, 9)
Gate shape: (16, 1)
Loss: 2.0272
Forward/backward time: 0.8 seconds
Peak allocated GPU memory: 1.94 GiB


In [ ]:
import time

# This is only a timing check. We will reinitialize the model before
# any actual experiment, so these updates won't be part of the final run.
optimizer = torch.optim.AdamW(gated_model.parameters(), lr=2e-5)
scaler = torch.amp.GradScaler("cuda")

gated_model.train()

torch.cuda.synchronize()
start = time.perf_counter()

for _ in range(20):
    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits, _ = gated_model(child_inputs, context_inputs, has_parent)
        loss = nn.CrossEntropyLoss()(logits, targets)

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(gated_model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

torch.cuda.synchronize()
seconds_per_batch = (time.perf_counter() - start) / 20

print(f"Average time per batch: {seconds_per_batch:.3f} seconds")
print(
    f"Rough training time per epoch: "
    f"{seconds_per_batch * ((len(splits.train) + 15) // 16) / 60:.1f} minutes"
)

Average time per batch: 0.155 seconds
Rough training time per epoch: 12.8 minutes


In [ ]:
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

gated_model = GatedDiscourseModel(
    "distilbert/distilbert-base-uncased",
    num_labels=len(splits.labels),
).to("cuda")

run_dir = Path("/content/gated_discourse_experiment_01")
run_dir.mkdir(parents=True, exist_ok=False)

label_to_id = {label: i for i, label in enumerate(splits.labels)}
batch_size = 16

def make_batch(examples):
    children = [ex.text for ex in examples]
    parents = [ex.metadata.get("parent_text", "") for ex in examples]

    child_inputs = tokenizer(
        children,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt",
    ).to("cuda")

    context_inputs = tokenizer(
        parents,
        children,
        padding=True,
        truncation="longest_first",
        max_length=256,
        return_tensors="pt",
    ).to("cuda")

    has_parent = torch.tensor(
        [bool(parent.strip()) for parent in parents],
        dtype=torch.float32,
        device="cuda",
    )

    targets = torch.tensor(
        [label_to_id[ex.label] for ex in examples],
        dtype=torch.long,
        device="cuda",
    )

    return child_inputs, context_inputs, has_parent, targets

train_loader = DataLoader(
    splits.train,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=list,
)

dev_loader = DataLoader(
    splits.dev,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=list,
)

optimizer = torch.optim.AdamW(gated_model.parameters(), lr=2e-5)
scaler = torch.amp.GradScaler("cuda")
criterion = nn.CrossEntropyLoss()

history = []
run_start = time.perf_counter()

for epoch in range(1, 4):
    gated_model.train()
    total_loss = 0.0

    for examples in tqdm(train_loader, desc=f"Epoch {epoch}/3"):
        child_inputs, context_inputs, has_parent, targets = make_batch(examples)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits, _ = gated_model(
                child_inputs, context_inputs, has_parent
            )
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(gated_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    gated_model.eval()
    predictions = []
    gold = []
    gate_values = []

    with torch.inference_mode():
        for examples in tqdm(dev_loader, desc="Development"):
            child_inputs, context_inputs, has_parent, targets = make_batch(examples)

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits, gates = gated_model(
                    child_inputs, context_inputs, has_parent
                )

            predictions.extend(logits.argmax(dim=1).cpu().tolist())
            gold.extend(targets.cpu().tolist())
            gate_values.extend(gates.squeeze(1).float().cpu().tolist())

    result = {
        "epoch": epoch,
        "train_loss": total_loss / len(train_loader),
        "dev_accuracy": accuracy_score(gold, predictions),
        "dev_macro_f1": f1_score(
            gold,
            predictions,
            labels=list(range(len(splits.labels))),
            average="macro",
            zero_division=0,
        ),
        "dev_weighted_f1": f1_score(
            gold, predictions, average="weighted", zero_division=0
        ),
        "mean_gate": float(np.mean(gate_values)),
    }

    history.append(result)

    torch.save(
        {
            "model_state_dict": gated_model.state_dict(),
            "labels": splits.labels,
            "model_name": "distilbert/distilbert-base-uncased",
            "max_length": 256,
            "epoch": epoch,
        },
        run_dir / f"epoch_{epoch}.pt",
    )

    (run_dir / "history.json").write_text(
        json.dumps(history, indent=2),
        encoding="utf-8",
    )

    print(
        f"\nEpoch {epoch}: "
        f"accuracy={result['dev_accuracy']:.4f}, "
        f"macro F1={result['dev_macro_f1']:.4f}, "
        f"weighted F1={result['dev_weighted_f1']:.4f}, "
        f"mean gate={result['mean_gate']:.3f}",
        flush=True,
    )

print(f"\nTotal time: {(time.perf_counter() - run_start) / 60:.1f} minutes")
print("Saved to:", run_dir)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/3:   0%|          | 0/4970 [00:00<?, ?it/s]

Development:   0%|          | 0/586 [00:00<?, ?it/s]


Epoch 1: accuracy=0.7509, macro F1=0.5530, weighted F1=0.7367, mean gate=0.448


Epoch 2/3:   0%|          | 0/4970 [00:00<?, ?it/s]

Development:   0%|          | 0/586 [00:00<?, ?it/s]


Epoch 2: accuracy=0.7460, macro F1=0.5630, weighted F1=0.7378, mean gate=0.466


Epoch 3/3:   0%|          | 0/4970 [00:00<?, ?it/s]

Development:   0%|          | 0/586 [00:00<?, ?it/s]


Epoch 3: accuracy=0.7451, macro F1=0.5893, weighted F1=0.7438, mean gate=0.502

Total time: 57.4 minutes
Saved to: /content/gated_discourse_experiment_01


In [ ]:
import json
import shutil
import torch
import transformers
from google.colab import files

tokenizer.save_pretrained(run_dir / "tokenizer")

(run_dir / "versions.json").write_text(
    json.dumps(
        {
            "torch": torch.__version__,
            "transformers": transformers.__version__,
        },
        indent=2,
    ),
    encoding="utf-8",
)

history_cells = get_ipython().history_manager.input_hist_raw

model_cells = [
    cell for cell in history_cells
    if "class GatedDiscourseModel(nn.Module):" in cell
    and "shutil.make_archive" not in cell
]

training_cells = [
    cell for cell in history_cells
    if "for epoch in range(1, 4):" in cell
    and "run_dir = Path(" in cell
    and "shutil.make_archive" not in cell
]

if not model_cells or not training_cells:
    raise RuntimeError(
        "Could not find the model or training cell in session history. "
        "Download the Colab notebook before continuing."
    )

(run_dir / "experiment_cells.py").write_text(
    model_cells[-1] + "\n\n" + training_cells[-1],
    encoding="utf-8",
)

archive = shutil.make_archive(
    "/content/gated_discourse_experiment_01",
    "zip",
    root_dir=run_dir,
)

print("Archive created:", archive)
files.download(archive)

Archive created: /content/gated_discourse_experiment_01.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
import pandas as pd
import torch

gated_model.eval()

all_gold = []
all_gates = []
all_has_parent = []
all_child_logits = []
all_context_logits = []

with torch.inference_mode():
    for examples in tqdm(dev_loader, desc="Inspecting development gates"):
        child_inputs, context_inputs, has_parent, targets = make_batch(examples)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            child_hidden = gated_model.encoder(
                **child_inputs
            ).last_hidden_state[:, 0]

            context_hidden = gated_model.encoder(
                **context_inputs
            ).last_hidden_state[:, 0]

            child_logits = gated_model.child_head(child_hidden)
            context_logits = gated_model.context_head(context_hidden)

            gate_input = torch.cat(
                [
                    child_hidden,
                    context_hidden,
                    has_parent.reshape(-1, 1),
                ],
                dim=1,
            )
            gates = torch.sigmoid(gated_model.gate(gate_input))

        all_gold.extend(targets.cpu().tolist())
        all_gates.extend(gates.squeeze(1).float().cpu().tolist())
        all_has_parent.extend(has_parent.cpu().tolist())
        all_child_logits.append(child_logits.float().cpu().numpy())
        all_context_logits.append(context_logits.float().cpu().numpy())

gold = np.array(all_gold)
gates = np.array(all_gates)
parent_present = np.array(all_has_parent, dtype=bool)
child_scores = np.concatenate(all_child_logits)
context_scores = np.concatenate(all_context_logits)

print("Gate distribution:")
print(
    pd.Series(gates).describe(
        percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
    )
)

print("\nMean gate with parent:", gates[parent_present].mean())
print("Mean gate without parent:", gates[~parent_present].mean())

score_sets = {
    "Child branch only": child_scores,
    "Equal mix": 0.5 * child_scores + 0.5 * context_scores,
    "Context branch only": context_scores,
    "Learned gate": (
        (1 - gates[:, None]) * child_scores
        + gates[:, None] * context_scores
    ),
}

print("\nDevelopment results with gate overrides:")
for name, scores in score_sets.items():
    pred = scores.argmax(axis=1)

    print(
        f"{name:20s} "
        f"accuracy={accuracy_score(gold, pred):.4f}  "
        f"macro F1={f1_score(gold, pred, labels=list(range(len(splits.labels))), average='macro', zero_division=0):.4f}"
    )

child_pred = child_scores.argmax(axis=1)
context_pred = context_scores.argmax(axis=1)

print(
    "\nBranch prediction disagreement:",
    f"{np.mean(child_pred != context_pred):.1%}",
)

Inspecting development gates:   0%|          | 0/586 [00:00<?, ?it/s]

Gate distribution:
count    9373.000000
mean        0.502079
std         0.223581
min         0.037384
5%          0.121106
25%         0.310059
50%         0.548340
75%         0.692383
95%         0.802246
max         0.919434
dtype: float64

Mean gate with parent: 0.5441422761191851
Mean gate without parent: 0.2507966586521694

Development results with gate overrides:
Child branch only    accuracy=0.5740  macro F1=0.4642
Equal mix            accuracy=0.7032  macro F1=0.5582
Context branch only  accuracy=0.5143  macro F1=0.2448
Learned gate         accuracy=0.7451  macro F1=0.5893

Branch prediction disagreement: 70.9%


In [ ]:
equal_pred = score_sets["Equal mix"].argmax(axis=1)
learned_pred = score_sets["Learned gate"].argmax(axis=1)

for name, mask in [
    ("Parent present", parent_present),
    ("Parent absent", ~parent_present),
]:
    equal_correct = equal_pred[mask] == gold[mask]
    learned_correct = learned_pred[mask] == gold[mask]

    print(f"\n{name}: {mask.sum():,} examples")
    print(f"  Equal-mix accuracy:   {equal_correct.mean():.4f}")
    print(f"  Learned-gate accuracy: {learned_correct.mean():.4f}")
    print(
        "  Corrected by learned gate:",
        int((~equal_correct & learned_correct).sum()),
    )
    print(
        "  Errors introduced by learned gate:",
        int((equal_correct & ~learned_correct).sum()),
    )


Parent present: 8,029 examples
  Equal-mix accuracy:   0.7022
  Learned-gate accuracy: 0.7467
  Corrected by learned gate: 580
  Errors introduced by learned gate: 223

Parent absent: 1,344 examples
  Equal-mix accuracy:   0.7091
  Learned-gate accuracy: 0.7359
  Corrected by learned gate: 70
  Errors introduced by learned gate: 34


In [ ]:
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
import json

checkpoint = torch.load(
    run_dir / "epoch_3.pt",
    map_location="cuda",
    weights_only=True,
)

assert checkpoint["labels"] == splits.labels

gated_model.load_state_dict(checkpoint["model_state_dict"])
gated_model.eval()

test_loader = DataLoader(
    splits.test,
    batch_size=16,
    shuffle=False,
    collate_fn=list,
)

test_gold = []
test_pred = []
test_records = []

with torch.inference_mode():
    for examples in tqdm(test_loader, desc="Evaluating test set"):
        child_inputs, context_inputs, has_parent, targets = make_batch(examples)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits, batch_gates = gated_model(
                child_inputs, context_inputs, has_parent
            )

        predictions = logits.argmax(dim=1).cpu().tolist()
        gold = targets.cpu().tolist()
        batch_gates = batch_gates.squeeze(1).float().cpu().tolist()

        test_pred.extend(predictions)
        test_gold.extend(gold)

        for ex, target, prediction, gate in zip(
            examples, gold, predictions, batch_gates
        ):
            test_records.append({
                "id": ex.id,
                "conversation_id": ex.metadata["conversation_id"],
                "gold": splits.labels[target],
                "prediction": splits.labels[prediction],
                "gate": gate,
            })

print("Test examples:", len(test_gold))
print("Accuracy:", accuracy_score(test_gold, test_pred))
print(
    "Macro F1:",
    f1_score(
        test_gold,
        test_pred,
        labels=list(range(len(splits.labels))),
        average="macro",
        zero_division=0,
    ),
)
print(
    "Weighted F1:",
    f1_score(test_gold, test_pred, average="weighted", zero_division=0),
)

print("\nPer-class results:")
print(
    classification_report(
        test_gold,
        test_pred,
        labels=list(range(len(splits.labels))),
        target_names=splits.labels,
        digits=4,
        zero_division=0,
    )
)

# Save individual predictions for later paired analysis.
test_output = run_dir / "test_predictions.jsonl"
with test_output.open("w", encoding="utf-8") as stream:
    for record in test_records:
        stream.write(json.dumps(record) + "\n")

print("Predictions saved:", test_output)

Evaluating test set:   0%|          | 0/647 [00:00<?, ?it/s]

Test examples: 10343
Accuracy: 0.7455283766798801
Macro F1: 0.5843609120842888
Weighted F1: 0.7444742138139496

Per-class results:
                  precision    recall  f1-score   support

       agreement     0.6749    0.5795    0.6236       516
    announcement     0.4152    0.5299    0.4656       134
          answer     0.8614    0.8271    0.8439      4304
    appreciation     0.8118    0.7372    0.7727       936
    disagreement     0.5263    0.2934    0.3768       409
     elaboration     0.5937    0.7036    0.6440      2004
           humor     0.3348    0.3108    0.3223       251
negativereaction     0.3476    0.3757    0.3611       173
        question     0.8230    0.8775    0.8494      1616

        accuracy                         0.7455     10343
       macro avg     0.5987    0.5816    0.5844     10343
    weighted avg     0.7493    0.7455    0.7445     10343

Predictions saved: /content/gated_discourse_experiment_01/test_predictions.jsonl


In [ ]:
from google.colab import files

files.download(str(test_output))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Gated Context Model for Discourse-Act Classification

## Objective

This experiment investigates whether a discourse classifier can learn how much
to rely on a comment's parent when predicting its discourse act.

The team's earlier experiments found that including the parent comment improved
classification overall. However, parent context did not help every example:
the paired development analysis found cases where the comment-only transformer
was correct and the parent-context transformer was not.

The question for this experiment is:

**Can a model that combines child-only and parent-context predictions through a
learned gate improve discourse classification?**

## Data and evaluation

The experiment uses the standardized Reddit Coarse Discourse Corpus. It follows
the existing baseline's filtering rules:

- Exclude synthetic placeholder utterances.
- Exclude comments without a discourse label.
- Exclude the `other` class.
- Exclude comments with empty cleaned text.

The remaining comments belong to nine discourse-act classes. Comments are
split by conversation using the repository's seed-42 split, so comments from
the same conversation do not appear in multiple partitions.

| Split | Labeled comments |
|---|---:|
| Training | 79,517 |
| Development | 9,373 |
| Test | 10,343 |

Development macro F1 was used to select the gated model's checkpoint.
The selected checkpoint was then evaluated on the test set.

## Model

The model uses a shared pretrained DistilBERT encoder in two branches:

1. **Child-only branch:** encodes the comment being classified.
2. **Context branch:** encodes the parent comment and the child together.

Each branch produces class scores. A learned gate combines those scores:

$$
z = g z_{\text{context}} + (1-g)z_{\text{child}},
$$

where $g\in[0,1]$ is computed separately for each example. The gate receives
both branch representations and an indicator of whether parent text is
available.

The context branch uses a shared maximum length of 256 tokens for the parent
and child. The tokenizer can allocate unused space from a shorter comment to
the longer one. The child-only branch also has a maximum length of 256 tokens.

The encoder and classification heads were trained jointly for three epochs
using ordinary cross-entropy loss. Training used a Tesla T4 GPU, a batch size
of 16, a learning rate of $2\times 10^{-5}$, mixed precision, and seed 42.

## Development results

The table compares the gated model with the team's reported parent-context
DistilBERT model and a separate shared-budget context experiment.

| Model | Development accuracy | Development macro F1 | Development weighted F1 |
|---|---:|---:|---:|
| Team's parent-context DistilBERT | 0.7362 | 0.5785 | 0.7365 |
| Shared-budget context model | 0.7317 | 0.5814 | 0.7346 |
| **Gated model** | **0.7451** | **0.5893** | **0.7438** |

The gated model's third epoch had the highest development macro F1 among its
three checkpoints and was selected for test evaluation.

The shared-budget model was trained separately. Its encoder changed both token
allocation and the earlier decode-and-retokenize processing step, so its
comparison with the team's original model does not isolate token allocation
alone.

## Gate analysis

The selected gated model's mean development gate was approximately 0.502.
However, individual gate values varied:

| Statistic | Gate value |
|---|---:|
| 5th percentile | 0.121 |
| 25th percentile | 0.310 |
| Median | 0.548 |
| 75th percentile | 0.692 |
| 95th percentile | 0.802 |

The mean gate was approximately **0.544** when parent text was present and
**0.251** when it was absent. The model receives an explicit parent-availability
indicator, so this difference is consistent with the gate responding to that
indicator. It does not establish why the model makes any individual prediction.

To inspect the trained model, its learned gate was overridden at inference
time while keeping the same branch weights:

| Inference setting | Development accuracy | Development macro F1 |
|---|---:|---:|
| Child-branch scores only | 0.5740 | 0.4642 |
| Equal 50–50 mix | 0.7032 | 0.5582 |
| Context-branch scores only | 0.5143 | 0.2448 |
| **Learned gate** | **0.7451** | **0.5893** |

Relative to the equal mix, the learned gate corrected 580 predictions and
introduced 223 errors among comments with parent text. Among comments without
parent text, it corrected 70 and introduced 34 errors.

These are inference-time overrides of the **same jointly trained model**.
The branches were not trained to act as standalone classifiers. A separately
trained fixed-mix model would be a stronger control for testing whether
learning the gate improves performance.

## Held-out test results

| Model | Test accuracy | Test macro F1 | Test weighted F1 |
|---|---:|---:|---:|
| Team's parent-context DistilBERT | 0.7376 | 0.5766 | 0.7387 |
| **Gated model** | **0.7455** | **0.5844** | **0.7445** |
| Difference | +0.0079 | +0.0078 | +0.0058 |

The gated model exceeds the team's reported result on all three aggregate
metrics in this comparison. These are results from one training run per model;
no confidence interval or paired significance test was calculated.

The team's result was previously reported on this test set. It was held out
from this experiment's training and checkpoint selection, but it is not an
unseen benchmark for the project as a whole.

### Per-class test F1

| Class | Support | Gated model F1 |
|---|---:|---:|
| Agreement | 516 | 0.6236 |
| Announcement | 134 | 0.4656 |
| Answer | 4,304 | 0.8439 |
| Appreciation | 936 | 0.7727 |
| Disagreement | 409 | 0.3768 |
| Elaboration | 2,004 | 0.6440 |
| Humor | 251 | 0.3223 |
| Negative reaction | 173 | 0.3611 |
| Question | 1,616 | 0.8494 |

Compared with the team's reported parent-context model, the gated model has
higher test F1 for humor (0.3223 versus 0.2957) and negative reaction
(0.3611 versus 0.2987). Its disagreement F1 is lower (0.3768 versus 0.4522).

## Limitations and next steps

- The aggregate test improvements are modest and come from a single training
  run. Training with additional seeds would help assess their stability.
- The teammate's individual predictions were not available for a paired
  comparison. Therefore, we cannot calculate a paired confidence interval for
  the difference between the two models.
- The gated model and the team's model differ in architecture and training
  implementation. The observed difference cannot be attributed to the gate
  alone.
- The fixed-gate analysis changes the gate only at inference time. Train a
  separate fixed-mix model to test whether a learned gate adds value.
- Gate values are descriptive model outputs. They should not be interpreted
  as definitive explanations of individual predictions.
- Disagreement F1 declined relative to the team's reported model. Future work
  should investigate this tradeoff without repeatedly tuning against the test
  set.
- The test set has already been used within the project. Evaluation on a new
  dataset would provide stronger evidence of generalization.